In [69]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

load_dotenv(override=True)

EMBEDDING_MODEL = "qwen3-embedding-8b"
QDRANT_COLLECTION = "accords_heures_supp"

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=os.environ["LLM_API_URL"],
    api_key=os.environ["LLM_API_KEY"],
    check_embedding_ctx_length=False
)

qdrant_client = QdrantClient(
    url=os.environ.get("QDRANT_API_URL", "http://qdrant:6333"),
    api_key=os.environ.get("QDRANT_API_KEY", None),
    timeout=60
)

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=QDRANT_COLLECTION,
    embedding=embeddings,
)

/tmp/ipykernel_6693/1671997699.py:20: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


In [70]:
from typing import Optional
from pydantic import BaseModel, Field

class ExtractionsHeuresSup(BaseModel):
    presence_contingent: bool = Field(
        description="True si un contingent d'heures supplémentaires est mentionné dans le document, sinon False"
    )
    contingent_heures: Optional[int] = Field(
        default=None,
        description="Le nombre d'heures du contingent d'heures supplémentaires (ex: 220), ou None si absent"
    )

In [92]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from qdrant_client.http import models

llm = ChatOpenAI(
    base_url=os.environ["LLM_API_URL"],
    api_key=os.environ["LLM_API_KEY"],
    model="gemma4-26b-moe",
    temperature=0
)
structured_llm = llm.with_structured_output(ExtractionsHeuresSup)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant spécialisé dans l'analyse de documents juridiques et RH. "
               "Analyse les extraits de texte fournis pour en extraire les informations demandées. "
               "Si l'information n'est pas présente dans les textes, indique False pour la présence."),
    ("human", "Voici les extraits du document :\n\n{context}\n\nExtrais les informations requises.")
])

def analyser_document(doc_id: str):
    filtre_document = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.doc_id",
                match=models.MatchValue(value=doc_id)
            )
        ]
    )

    retriever = vector_store.as_retriever(
        search_kwargs={
            "k": 4,
            "filter": filtre_document
        }
    )
    
    query = "contingent d'heures supplémentaires"
    docs = retriever.invoke(query)

    context_text = "\n\n".join([d.page_content for d in docs])
    
    # Exécution de la chaîne
    chain = prompt | structured_llm
    resultat = chain.invoke({"context": context_text})
    
    return resultat

In [97]:
from pydantic import BaseModel, Field


class DetailContingent(BaseModel):
    categorie_ou_population: str | None = Field(
        default=None,
        description="Population concernée si le contingent est spécifique (ex: 'Non-cadres', 'Salariés en modulation', 'Tous les salariés').",
    )
    nombre_heures: int = Field(
        description="Nombre d'heures du contingent d'heures supplémentaires fixé par l'accord."
    )
    periode: str | None = Field(
        default="annuel",
        description="Période de référence du contingent (ex: 'annuel', 'par exercice').",
    )


class ExtractionContingentHeuresSup(BaseModel):
    presence_contingent_conventionnel: bool = Field(
        description=(
            "True uniquement si le document fixe un contingent d'heures supplémentaires "
            "propre à l'entreprise. False si le document est muet ou ne fait que citer le contingent légal de référence."
        )
    )
    contingents: list[DetailContingent] = Field(
        default_factory=list,
        description="Liste des contingents d'heures supplémentaires définis spécifiquement par l'accord.",
    )
    citation_exacte: str | None = Field(
        default=None,
        description="Extrait exact du texte traitant du contingent d'heures supplémentaires. Obligatoire si un contingent est trouvé.",
    )
    justification_ou_reserve: str | None = Field(
        default=None,
        description="Remarques éventuelles (ex: rappel du droit commun, contingent variable, proratisation).",
    )

In [98]:
SYSTEM_PROMPT = """
Tu es un juriste expert en droit du travail français, spécialisé dans l'analyse d'accords d'entreprise et de conventions collectives.

Ton unique mission est d'analyser le texte extrait d'un accord d'entreprise et d'extraire la valeur du **contingent d'heures supplémentaires** fixé par cet accord.

### Définition juridique précise
Le **contingent d'heures supplémentaires** est le seuil annuel d'heures supplémentaires qu'un employeur peut faire effectuer par un salarié sans avoir à demander l'avis du comité social et économique (CSE) ni à accorder la contrepartie obligatoire en repos (COR).

### RÈGLES STRICTES D'EXTRACTION

1. **Source exclusive :** Base-toi UNIQUEMENT sur les extraits de texte fournis. N'utilise aucune connaissance externe pour déduire un chiffre qui ne figure pas explicitement dans le texte.
2. **Contingent conventionnel vs Contingent légal :**
   - Si le document fixe un nombre d'heures spécifique à l'entreprise (ex. "Le contingent d'heures supplémentaires est fixé à 180 heures"), extrais ce nombre.
   - Si le document se contente de rappeler le contingent légal (ex. "Conformément à la loi, le contingent est de 220 heures") sans créer de règle propre à l'entreprise, mets `presence_contingent_conventionnel` à `False` et explique-le dans `justification_ou_reserve`.
3. **Absence ou doute :** Si le texte ne mentionne pas clairement un chiffre pour le contingent d'heures supplémentaires, renvoie `presence_contingent_conventionnel = False` et `contingents = []`. Ne devine jamais un chiffre.
4. **Distinction stricte des concepts :** Ne confonds PAS le contingent d'heures supplémentaires avec :
   - Le plafond annuel d'heures de travail (ex. 1607 heures).
   - Le plafond de jours pour un forfait annuel en jours (ex. 218 jours).
   - Les heures complémentaires (salariés à temps partiel).
   - La limite maximale hebdomadaire de travail (ex. 48 heures).
5. **Preuve par citation :** Quand un contingent est identifié, recopie le passage exact du texte dans `citation_exacte`.
"""

In [99]:
USER_PROMPT = """
Analyse les extraits de l'accord d'entreprise ci-dessous et extrait les informations relatives au contingent d'heures supplémentaires selon le schéma JSON demandé.

### Extraits du document :
---
{context}
---

Si le texte ne contient aucun contingent d'heures supplémentaires conventionnel, réponds en mettant `presence_contingent_conventionnel` à `False`.
"""

In [ ]:
QUERY = """Query: contingent d'heures supplémentaires"""

In [100]:
structured_llm = llm.with_structured_output(ExtractionContingentHeuresSup)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_PROMPT)
])

def analyser_document_expert(doc_id: str):
    filtre_document = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.doc_id",
                match=models.MatchValue(value=doc_id)
            )
        ]
    )

    retriever = vector_store.as_retriever(
        search_kwargs={
            "k": 5,
            "filter": filtre_document
        }
    )
    
    query = QUERY
    docs = retriever.invoke(query)

    context_text = "\n\n".join([d.page_content for d in docs])
    
    # Exécution de la chaîne
    chain = prompt | structured_llm
    resultat = chain.invoke({"context": context_text})
    
    return resultat

In [101]:
extraction = analyser_document("ACCOTEXT000041680349")

In [ ]:
extraction.contingents

[DetailContingent(categorie_ou_population='l’ensemble des salariés concernés (personnel Ouvrier et Etam de chantier dont la durée de travail est décomptée en heures)', nombre_heures=400, periode='par an')]

In [84]:
extraction[1]

['ACCOTEXT000041680349',
 'ACCOTEXT000049480862',
 'ACCOTEXT000048348517',
 'ACCOTEXT000049479453']